# Low-Effort Optimization Playbook

These five levers change the model request: which model runs it, what you ask, how much it can generate, and which processing can be reused. Each is useful on its own; begin with the smallest change that addresses a measured problem.

| Lever | What it changes | Question to answer |
|---|---|---|
| 01 Model selection | Model size/capability | Can a smaller model meet the task's quality bar? |
| 02 Prompt design, provider by provider | Instructions and examples | Is the task clear enough to answer consistently? |
| 03 Output and sampling controls | Generation controls | Can the answer be focused without truncation? |
| 04 Prompt caching, provider by provider | Repeated stable input | Does reuse outweigh the initial write cost? |
| 05 Reasoning effort and budgets | Work before/during the answer | Does more reasoning improve this task? |

Read each task before executing it, predict which metric should move, then compare the answer as well as the measurements.

## Setup

Use **Bedrock Workshop (Python 3.13)** from the [repository setup](../README.md), with a fresh kernel. Dependencies and shared infrastructure, including Langfuse, are prepared before the lessons. AWS-native telemetry (`OBSERVABILITY_BACKEND=agentcore`) is the default; `langfuse` is a ready alternative and `both` exports to both. Select the backend before initialization and restart the kernel when switching.

Run the examples in order. Model and evaluator calls incur charges; optional exercises are gated in code. The shared model/pricing catalog is dated **2026-09-24**. Keep the task fixed and change only the lever being tested; outside model selection, keep the selected model/API fixed too.

`run_case` below sends one native `bedrock-runtime.converse` request and records its response. The observation context adds telemetry; it does not call the model. These are direct model requests. The managed Amazon Bedrock Agents product is a separate service; AgentCore is used here for observability and the optional response evaluation.


In [ ]:
import hashlib
import json
import os
import sys
import time
import uuid
from datetime import UTC, datetime, timezone
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from workshop_utils.bedrock import build_converse_request, normalize_usage
from workshop_utils.models import ModelRegistry, Support, caps, resolve_model
from workshop_utils.observability import (
    converse_observation,
    resolve_notebook_backend,
    response_span,
    setup_observability,
)
from workshop_utils.pacing import attach_boto3, get_workshop_pacer
from workshop_utils.pricing import AmbiguousCacheUsageError, InferredPriceError, UnknownPriceError, calculate_cost

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BACKEND = resolve_notebook_backend(REPO_ROOT / ".env")
SDK_CONFIG = Config(connect_timeout=10, read_timeout=120,
                    retries={"mode": "standard", "total_max_attempts": 3})
# Resolve the existing stack's destination before initializing any exporters.
# An explicit environment value supports event/own-account setup without an SSM read.
if BACKEND in {"agentcore", "both"}:
    NOTEBOOK_LOG_GROUP = os.environ.get("WORKSHOP_NOTEBOOK_LOG_GROUP")
    if not NOTEBOOK_LOG_GROUP:
        setup_ssm = boto3.client("ssm", region_name=REGION, config=SDK_CONFIG)
        try:
            NOTEBOOK_LOG_GROUP = setup_ssm.get_parameter(
                Name="/app/customersupport/observability/notebook_log_group"
            )["Parameter"]["Value"]
        finally:
            setup_ssm.close()
    os.environ["WORKSHOP_NOTEBOOK_LOG_GROUP"] = NOTEBOOK_LOG_GROUP
OBS = setup_observability(BACKEND, service_name="playbook-low", region=REGION)
print({"backend": OBS.backend, "logs_metrics_log_group": OBS.log_group})
RUNTIME = boto3.client("bedrock-runtime", region_name=REGION, config=SDK_CONFIG)
attach_boto3(RUNTIME, get_workshop_pacer())
AGENTCORE = boto3.client("bedrock-agentcore", region_name=REGION, config=SDK_CONFIG)
registry = ModelRegistry(region=REGION, overrides={
    key: os.environ[env] for key, env in {
        "small": "PLAYBOOK_SMALL_MODEL_ID", "workhorse": "PLAYBOOK_WORKHORSE_MODEL_ID",
        "deep": "PLAYBOOK_DEEP_MODEL_ID", "gpt-small": "PLAYBOOK_GPT_MODEL_ID",
    }.items() if os.environ.get(env)
})
SMALL = registry.resolve("small").model_id
WORKHORSE = registry.resolve("workhorse").model_id
DEEP = registry.resolve("deep").model_id
GPT = registry.resolve("gpt-small").model_id
RUN_GPT = os.environ.get("RUN_GPT_COMPARISON", "0") == "1"
RUN_EVAL = os.environ.get("RUN_AGENTCORE_EVALUATION", "0") == "1"
RUN_ID = uuid.uuid4().hex[:12]
RUN_ROWS = []

def supported(model_id, *features):
    unavailable = {f: caps(model_id).support(f).value for f in features
                   if caps(model_id).support(f) != Support.SUPPORTED}
    if unavailable:
        print({"skipped_model": model_id, "capabilities": unavailable})
    return not unavailable

def run_case(model_id, query=None, *, label, messages=None, **controls):
    """A validated call, returned usage, and a local response-only observation."""
    model = resolve_model(model_id, region=REGION)
    if messages is None:
        messages = [{"role": "user", "content": [{"text": query}]}]
    request = build_converse_request(model, messages, **controls)
    sid, tid = str(uuid.uuid4()), uuid.uuid4().hex
    t0 = time.time_ns()
    # One native Bedrock Runtime call; the context records this same response.
    with converse_observation(request, session_id=sid) as record_response:
        response = RUNTIME.converse(**request)
        response = record_response(response)
    t1 = time.time_ns()
    text = "".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b)
    ttl = model.capabilities.cache_ttls[0] if model.capabilities.cache_mode == "implicit" else None
    points = [b["cachePoint"] for b in request.get("system", []) if "cachePoint" in b]
    points += [b["cachePoint"] for m in messages for b in m["content"] if "cachePoint" in b]
    ttls = {point.get("ttl", "5m") for point in points}
    if len(ttls) == 1:
        ttl = next(iter(ttls))
    usage = normalize_usage(response["usage"], source="converse", cache_ttl=ttl)
    cost = None
    price_note = ""
    requested_tier = controls.get("service_tier", "standard")
    served_tier = response.get("serviceTier", {}).get("type")
    pricing_tier = ("standard" if served_tier == "default" else served_tier) or (
        "standard" if requested_tier == "standard" else None)
    try:
        if pricing_tier is None:
            raise UnknownPriceError("No served service tier was returned; do not assume the requested discount.")
        cost = calculate_cost(model, usage, region=REGION, service_tier=pricing_tier).as_dict()
        price_note = "; ".join(cost.get("price_notes", []))
    except (UnknownPriceError, InferredPriceError, AmbiguousCacheUsageError) as exc:
        price_note = str(exc)  # Leave unknown/inferred cost unpriced, never zero.
    sdk_retries = response.get("ResponseMetadata", {}).get("RetryAttempts", 0)
    returned_response_cost = cost["total_cost"] if cost else None
    if sdk_retries:
        price_note = (price_note + " SDK retries occurred; usage from earlier attempts is unknown.").strip()
    ids = response.get("_trace", {})
    observed = response_span(
        query or json.dumps(messages), text, session_id=ids.get("session_id", sid),
        trace_id=ids.get("trace_id", tid), span_id=ids.get("span_id", uuid.uuid4().hex[:16]),
        start_time_ns=t0, end_time_ns=t1, service_name="playbook-low",
    ) if text else None
    provider_fields = request.get("additionalModelRequestFields", {})
    effort_config = provider_fields.get("output_config", provider_fields.get("reasoning", {}))
    row = {
        "run_id": RUN_ID, "label": label, "recorded_at": datetime.now(UTC).isoformat(),
        **model.as_dict(), **usage.as_dict(), "text": text,
        "latency_ms": (t1-t0)/1e6, "ttft_ms": None,
        "stop_reason": response.get("stopReason"), "served_tier": response.get("serviceTier"),
        "sdk_retries": sdk_retries, "usage_complete": sdk_retries == 0,
        "cost_usd": returned_response_cost if not sdk_retries else None,
        "returned_response_cost_usd": returned_response_cost, "price": cost, "price_note": price_note,
        "request": request, "response": response, "span": observed,
        "sdk_version": boto3.__version__, "dataset_version": "playbook-2026-09-24",
        "effective_effort": effort_config.get("effort", model.capabilities.default_effort),
        "request_sha256": hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest(),
    }
    RUN_ROWS.append(row)
    return row

def show(row):
    print({k: row[k] for k in ("label", "model_id", "text", "input_tokens", "output_tokens",
                                "cache_read_tokens", "cache_write_tokens", "latency_ms", "stop_reason",
                                "cost_usd", "price_note")})

for alias in ("small", "workhorse", "deep", "gpt-small"):
    print(registry.resolve(alias).as_dict())
print("Resolved offline; this does not prove account access or feature execution.")


## 1. Model Selection — Lever 01

The highest-leverage cost decision is **which model runs each call**. A straightforward extraction or classification may be well within a small model's capabilities; ambiguous troubleshooting may need a stronger one. The price per token is only part of that choice: failed answers, retries, and additional tool calls also cost money.

**Start small, escalate on evidence.** The next cell runs the same labelled tasks on Haiku 4.5 and Sonnet 5, with an optional GPT-5.6 Luna comparison. Inspect each answer and its `correct` flag in `selection_rows`, alongside `LABELLED_TICKETS`, before interpreting the printed cost summary. A model that passes one easy example may still fail a boundary case, so keep some inputs out of prompt tuning.

Re-benchmark when changing model generations. A task that needed the larger model last year may now fit the smaller one; a past result is a starting hypothesis, not a permanent routing rule.

Compare the same labelled task set on Haiku 4.5, Sonnet 5 and optional GPT-5.6 Luna. Report cost per solved task, counting every attempt. Unpriced rows stay unknown. API acceptance, task success and capability support are separate. Temperature is not a universal control.

| Model class | Start with | Escalate when |
|---|---|---|
| Small / fast | Classification, extraction, short summaries | Labels or essential facts are missed |
| Workhorse | Tool use and support answers combining several facts | Multi-step tasks still fail with clear instructions |
| Frontier | Difficult reasoning, coding, or fact-critical synthesis | Evaluation shows the larger model improves the result |

Check feature support as well as price. The same text can tokenize differently across models, so use each response’s actual usage. Compare the selected inference profile at current rates. Model choice and reasoning effort work together; Section 5 checks whether additional effort is worth its cost.


In [ ]:
LABELLED_TICKETS = [
    ("I was charged twice.", "billing"),
    ("My headset cannot connect over Bluetooth.", "technical"),
    ("How do I send back my unopened keyboard?", "returns"),
    ("Please correct the address on the invoice.", "billing"),
]
selection_rows = []
for model_id in [SMALL, WORKHORSE] + ([GPT] if RUN_GPT else []):
    for ticket, expected in LABELLED_TICKETS:
        r = run_case(model_id, "Classify as billing, technical, or returns. Return only the label. " + ticket,
                     label="L01-selection", max_tokens=1024)
        r["correct"] = r["text"].strip() == expected and r["stop_reason"] == "end_turn"
        selection_rows.append(r)
    rows = [r for r in selection_rows if r["model_id"] == model_id]
    # Inspect an existing request; displaying it makes no additional model call.
    print({"Converse request": rows[0]["request"]})
    solved = sum(r["correct"] for r in rows)
    total = sum(r["cost_usd"] for r in rows) if all(r["cost_usd"] is not None for r in rows) else None
    print({"model": model_id, "solved": solved, "n": len(rows), "model_cost": total,
           "model_cost_per_solved": total / solved if total is not None and solved else None})

## 2. Prompt Design, Provider by Provider — Lever 02

Hedging, boilerplate, and vague instructions cost tokens on every call and can make the task harder to follow. Three techniques, in order of how often you reach for them:

1. **Clear, specific instructions** — the baseline, always
2. **Few-shot examples** — when instructions alone leave room to guess
3. **Structured output** — when downstream code consumes the result

### 2.1 Clear, specific instructions

A well-designed prompt is **directive** (says exactly what to do, leading with the verb), **sectioned** (structure the model can follow), and **parsable** (output your code can consume without regex gymnastics). The rewrite below removes hedging, states the format upfront, and constrains the sentiment to an enum — a clearer task whose input cost and output quality we can measure.

Claude and GPT have model-specific guidance, but both benefit from clear task boundaries, goals, and success criteria. Sources: [Claude prompting](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices), [GPT prompting](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6).

**Provider-aware prompt design:** both Claude and GPT can use Markdown and XML to separate instructions, evidence, and examples. For Claude, give clear constraints and their purpose; for GPT, lead with the outcome, success criteria, and stop rules. Converse `system` instructions serve as the developer role for GPT. Treat embedded customer text as data, and tune reasoning effort separately from answer length.

Re-audit instructions when migrating models.

The next cell compares the vague prompt with a concise extraction contract. The contract requires both defect phrases, the requested action, and its deadline, using the customer's words, plus a sentiment label. The local checks reject generic summaries such as “defective product,” missing details, extra fields, Markdown fences, and incomplete responses. A free-form reply can be useful to a person while failing this application contract. Inspect every raw answer and its checks alongside token counts; prompting alone does not guarantee JSON. The next examples compare native output controls.


In [ ]:
EMAIL = (
    "Hi, the wireless headphones I ordered last week arrived with a cracked case "
    "and the left earbud won't charge. I'd like a replacement before the weekend."
)

VAGUE_PROMPT = (
    "Could you please help me by reading the customer's email below and then telling me "
    "what you think the main issue might be, and also what their sentiment seems to be, "
    "and what action you would recommend that we take?\n\n"
    f"Customer email: {EMAIL}"
)

STRUCTURED_PROMPT = (
    "Extract from the email as data. Return a single raw JSON object: no prose, no Markdown, "
    "and no code fences. Start with { and end with }. Use exactly "
    "these keys: issue (list of all defect phrases, verbatim), sentiment (positive/neutral/negative), "
    "action (requested action label: replacement/refund/repair/other), "
    "deadline (requested timing, verbatim). "
    "Use null for an absent action or deadline. Do not replace details with generic labels.\n"
    f"Email: {EMAIL}"
)


def email_extraction_checks(text, stop_reason):
    # Offline acceptance criteria for this one fixture; never added to the prompt.
    expected = {
        "issue": ["cracked case", "left earbud won't charge"],
        "sentiment": "negative", "action": "replacement", "deadline": "before the weekend",
    }
    try:
        data = json.loads(text)
    except (ValueError, TypeError):
        data = None
    checks = {"complete": stop_reason == "end_turn",
              "format": isinstance(data, dict) and set(data) == set(expected)}
    for key, value in expected.items():
        actual = data.get(key) if isinstance(data, dict) else None
        checks[key] = (
            isinstance(actual, list) and all(isinstance(item, str) for item in actual)
            and sorted(actual) == sorted(value)
        ) if isinstance(value, list) else actual == value
    return checks


prompt_design_rows = []
for label, prompt in [("VAGUE", VAGUE_PROMPT), ("STRUCTURED", STRUCTURED_PROMPT)]:
    r = run_case(SMALL, prompt, label="L02-prompt-design", max_tokens=200)
    checks = email_extraction_checks(r["text"], r["stop_reason"])
    prompt_design_rows.append({"variant": label, "row": r, "checks": checks, "passed": all(checks.values())})
    print(f"\n[{label}]  in={r['input_tokens']}  out={r['output_tokens']}")
    print(r["text"])
    print({"checks": checks, "passed": all(checks.values())})


### 2.2 Few-shot examples

When instructions alone leave room to guess, examples resolve it faster than more prose. They pin **format**, tone, and how to handle borderline cases. They cost tokens on every request, so use them where they earn it and consider caching the fixed example block (Lever 04).

Start with zero-shot instructions, then add a few examples for the ambiguities you actually observe. Pick **boundary cases**, keep them **diverse**, and keep classification examples **class-balanced**. Use a consistent output structure and correct labels.

The next cell compares zero-shot and few-shot product extraction. Check all four fields in `PRODUCT|BRAND|PRICE|CATEGORY`, including normalized prices, and compare consistency across the products.

If examples vary by request, keep the fixed core before the cache boundary and put changing examples after it.


In [ ]:
# Zero-shot vs few-shot: extract product info into one exact pipe-delimited format.
# Run across several inputs to see CONSISTENCY, not just one lucky output.
PRODUCTS = [
    "Apple MacBook Pro 16-inch with M3 chip, 32GB RAM, Space Black - $2,499",
    "Sony WH-1000XM5 wireless noise cancelling headphones in silver for $348",
    "Samsung 65 inch OLED 4K Smart TV (2024 model) priced at $1,799.99",
]

ZERO_SHOT = """Extract product info in this exact format:
PRODUCT|BRAND|PRICE|CATEGORY

Product: "{product}"
Output:"""

# 3 boundary/diverse, class-balanced, byte-identical examples
FEW_SHOT = """Extract product info in this exact format:
PRODUCT|BRAND|PRICE|CATEGORY

Examples:
Product: "Dell XPS 15 laptop with Intel i7, 16GB memory - $1,299"
Output: XPS 15|Dell|1299|laptop
Product: "Bose QuietComfort earbuds, noise cancelling, $279 retail"
Output: QuietComfort Earbuds|Bose|279|audio
Product: "LG 55\\" C3 OLED TV (2023) on sale for $1,196.99"
Output: C3 OLED 55"|LG|1196.99|tv

Now extract:
Product: "{product}"
Output:"""

# For each product we print the zero-shot output then the few-shot output, so you can
# compare complete outputs; validate whether few-shot actually improves the format.
for p in PRODUCTS:
    outs = {}
    for label, tmpl in [("zero", ZERO_SHOT), ("few", FEW_SHOT)]:
        r = run_case(SMALL, tmpl.format(product=p), label="L02-prompt-design", max_tokens=60)
        outs[label] = r["text"].strip()
    print(f"\n{p[:60]}")
    print(f"  {'zero-shot:':<12}{outs['zero']}")
    print(f"  {'few-shot:':<12}{outs['few']}")

### Output methods depend on the model/API

If downstream code consumes the answer, “almost valid JSON” is a real failure: prose around the object, a missing key, or the wrong type can break a parser. There are three distinct approaches:

| Approach | How it works | What to inspect |
|---|---|---|
| Prompted JSON | Instructions ask for a shape | Parseability, required fields, facts |
| Tool use | Model proposes a function call with arguments | Chosen tool, argument schema, business constraints |
| Native structured output | A supported API constrains output to a JSON schema | Schema conformance plus factual correctness |

Tool use is appropriate when the model will call a function; native structured output suits pure extraction where supported. A schema can enforce shape, but it cannot prove that an extracted price or sentiment is correct.

**Assistant prefill** (starting the answer with text such as `{`) is not a general recipe: Sonnet 5 and Opus 5 do not support it. Availability depends on the model, endpoint, and API. Haiku 4.5 and GPT-5.6 Luna support the native-schema exercises here; Sonnet 5 uses explicit format instructions and application validation on Converse. Forced tool selection does not itself enforce a strict schema. [Bedrock structured output](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html).

The following experiment checks each capability separately. Report refusal, truncation, invalid JSON and wrong facts as outcomes.

All three methods extract the same customer report: retain both the cracked screen and missing charger, negative sentiment, and the requested `replacement`. Here `action` is a customer-request label, never approval or an instruction to ship a replacement. Field descriptions ask for specific defect phrases instead of generic categories.

Read the separate `schema_valid`, `semantic_checks`, and `method_checks` results for every method/model. A forced-tool result must contain exactly one `classify_email` proposal and finish with `tool_use`; zero or wrong calls fail. Text responses must finish with `end_turn`. Fenced prompted JSON remains a visible format failure; native schema validity does not guarantee correct facts. `output_method_rows` retains each original row and quality result without repair or extra calls. These are narrow fixture checks: also inspect raw outputs for additional unsupported claims.

Validate arguments before a real tool executes. When native schemas are unavailable, use explicit instructions with application validation and bounded repair/retry, or select a supported model. Include those extra calls in the task cost.

The editable schema and tool configuration below become Converse `outputConfig` and `toolConfig`; inspect the printed request fields. This tool-use example validates proposed arguments without executing a function. An agent framework such as Strands is useful when you add tool execution and feed tool results back into a model loop.


In [ ]:
CLASSIFY_EMAIL = "My order #A-123 arrived with a cracked screen and the charger is missing. I want a replacement."
EXTRACTION_RULES = (
    "Extract only what the customer reports and requests. Preserve every specific defect. "
    "The action field labels the customer's request, not a recommendation, approval, "
    "or instruction to fulfil it. Do not claim that any action has been taken."
)
SCHEMA = {
    "type": "object", "properties": {
        "issue": {"type": "string", "description": (
            "Every reported defect as a short, specific phrase grounded in the customer's email. "
            "Keep the affected component and its problem together; separate defects with '; '. "
            "Do not replace specific defects with generic categories."
        )},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"],
                      "description": "The customer's sentiment expressed in the email."},
        "action": {"type": "string", "enum": ["replacement", "refund", "repair", "other", "none"],
                   "description": (
                       "The customer-requested outcome only, never approval or a fulfilment instruction. "
                       "Use other for a different explicit request, or none when no outcome is requested."
                   )},
    }, "required": ["issue", "sentiment", "action"], "additionalProperties": False,
}
from jsonschema import ValidationError, validate

output_method_rows = []


def inspect_output(row, *, tool_calls=None):
    is_tool = tool_calls is not None
    data, schema_error = None, None
    try:
        if is_tool:
            data = tool_calls[0].get("input") if len(tool_calls) == 1 else None
        else:
            data = json.loads(row["text"])
        validate(data, SCHEMA)
    except (json.JSONDecodeError, ValidationError) as exc:
        schema_error = str(exc)
    # Separate, offline checks for this fixture; these values are not sent as answers.
    fields = data if isinstance(data, dict) else {}
    issue = fields.get("issue")
    issue_parts = {part.strip().casefold().rstrip(".") for part in issue.split(";")} if isinstance(issue, str) else set()
    semantic_checks = {
        "cracked_screen": bool(issue_parts & {"cracked screen", "screen cracked", "screen is cracked", "the screen is cracked"}),
        "missing_charger": bool(issue_parts & {"missing charger", "charger missing", "charger is missing", "the charger is missing"}),
        "negative_sentiment": fields.get("sentiment") == "negative",
        "requested_replacement": fields.get("action") == "replacement",
    }
    method_checks = {
        "complete": row["stop_reason"] == ("tool_use" if is_tool else "end_turn"),
        "one_expected_tool": not is_tool or (
            len(tool_calls) == 1 and tool_calls[0].get("name") == "classify_email"
        ),
    }
    quality = {
        "method": row["label"], "model_id": row["model_id"], "stop": row["stop_reason"],
        "tool_calls": len(tool_calls) if is_tool else None,
        "schema_valid": schema_error is None, "schema_error": schema_error,
        "semantic_checks": semantic_checks, "semantic_valid": all(semantic_checks.values()),
        "method_checks": method_checks,
        "passed": schema_error is None and all(semantic_checks.values()) and all(method_checks.values()),
        "data": data, "raw": tool_calls if is_tool else row["text"],
    }
    output_method_rows.append({"row": row, **quality})
    print(quality)  # Raw failures stay visible; no stripping fences, repair, or retry.


r = run_case(SMALL, CLASSIFY_EMAIL, label="L02-prompted-JSON", max_tokens=512,
             system=[{"text": EXTRACTION_RULES + " Reply with only JSON matching this schema: " + json.dumps(SCHEMA)}])
inspect_output(r)
if supported(SMALL, "tools", "forced_tool_choice"):
    r = run_case(SMALL, CLASSIFY_EMAIL, label="L02-tool-choice", max_tokens=512,
        system=[{"text": EXTRACTION_RULES}],
        tool_config={"tools": [{"toolSpec": {
            "name": "classify_email", "description": "Extract reported defects and the customer's requested outcome; no side effects.",
            "inputSchema": {"json": SCHEMA}}}], "toolChoice": {"tool": {"name": "classify_email"}}})
    print({"toolConfig": r["request"]["toolConfig"]})
    tool_calls = [b["toolUse"] for b in r["response"]["output"]["message"]["content"] if "toolUse" in b]
    inspect_output(r, tool_calls=tool_calls)
for model_id in [SMALL, WORKHORSE] + ([GPT] if RUN_GPT else []):
    if supported(model_id, "structured_output"):
        r = run_case(model_id, CLASSIFY_EMAIL, label="L02-native-schema", max_tokens=1024,
            system=[{"text": EXTRACTION_RULES}],
            output_config={"textFormat": {"type": "json_schema", "structure": {"jsonSchema": {
                "name": "classify_email", "schema": json.dumps(SCHEMA)}}}})
        print({"outputConfig": r["request"]["outputConfig"]})
        inspect_output(r)


## 3. Output and Sampling Controls — Lever 03

**`max_tokens` is the maximum allowed output, not a target the model must fill.** A model can finish naturally before the cap. The exercise compares caps so you can distinguish shorter completed answers from answers cut off mid-sentence. On reasoning models, the allowance can include reasoning as well as visible text.

**Stop sequences** end generation at a chosen marker. They are useful with a real output terminator, but a common string such as a Markdown heading can stop a valid answer early. **Temperature**, where supported, controls sampling variability; lowering it does not guarantee factual accuracy or identical outputs. Claude 5 uses supported effort controls instead of accepting a copied temperature setting.

Predict which change affects output tokens, which affects variation, and which changes only the reserved quota if the answer already ends early. Then read the actual stop reason and usage.

Compare output caps, temperature, and stop sequences in the cells below. The output cap includes reasoning. Initial quota reservation is input plus maximum output; completion accounting applies the model's output burndown. **Burndown is not a price multiplier.** It is 5× for Haiku 4.5/Sonnet 4.6, 15× for Opus 4.8, and 10× for Sonnet 5/Opus 5/5.5 and documented GPT-5.6/runtime paths. Unknown factors remain unknown.

Sources: [Quota accounting](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas-token-burndown.html) · [Model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters.html).


### max_tokens: leave room for the answer

Compare caps of 4,096 and 256 tokens on the same summary task. Check whether both answers finish, then compare actual usage with the quota reservation estimates.


In [ ]:
EXAMPLE_TASK = "Summarize in one sentence: Delivery was fast, but the device failed after two weeks."
for maximum in (4096, 256):
    r = run_case(SMALL, EXAMPLE_TASK, label="L03-output-cap", max_tokens=maximum)
    print({"inferenceConfig": r["request"]["inferenceConfig"]})
    factor = caps(SMALL).output_quota_factor
    print({"max_tokens": maximum, "text": r["text"], "stop_reason": r["stop_reason"],
           "initial_reservation_estimate": r["input_tokens"] + r["cache_write_tokens"] + maximum,
           "completion_quota_estimate": r["input_tokens"] + r["cache_write_tokens"] + factor*r["output_tokens"]
               if factor is not None else None,
           "actual_output": r["output_tokens"],
           "latency_ms": r["latency_ms"], "cost_usd": r["cost_usd"]})


### temperature: compare variation

Generate a coffee-shop tagline three times at each temperature. Compare useful variety, relevance, and length; temperature zero does not guarantee identical answers.


In [ ]:
if supported(SMALL, "temperature"):
    for temperature in (0.0, 0.7, 1.0):
        for repeat in range(3):
            r = run_case(SMALL, "Write a one-sentence tagline for a coffee shop.",
                         label=f"L03-temperature-{temperature}-{repeat}",
                         max_tokens=256, temperature=temperature)
            if repeat == 0:
                print({"inferenceConfig": r["request"]["inferenceConfig"]})
            show(r)


### stop_sequences: end at a known boundary

Compare the same request with and without `###STOP` as a stop sequence. Inspect the returned text, stop reason, and token usage.


In [ ]:
if supported(SMALL, "stop_sequences"):
    for use_stop in (False, True):
        r = run_case(SMALL, "Reply with billing then ###STOP then a short explanation.",
                     label=f"L03-stop-{use_stop}", max_tokens=256,
                     **({"stop_sequences": ["###STOP"]} if use_stop else {}))
        print({"inferenceConfig": r["request"]["inferenceConfig"]})
        show(r)


## 4. Prompt Caching, Provider by Provider — Lever 04

Most requests resend the same system instructions, examples, or product documents. **Prompt caching stores the processed prefix** so the next request can reuse it. The reusable portion stays fixed; the new customer question goes after it.

The demo uses a product manual as a stable prefix and asks different questions. On an eligible cache miss, inspect the write counter. On later hits, inspect the read counter and the remaining uncached input. A tool-using agent may read that prefix more than once within one customer request because it calls the model again after a tool result.

Placement follows **tools → system → messages**: static content first, dynamic content last. Editing the cached prefix or exceeding its time-to-live (TTL) can require another write. A one-off request may pay the premium without getting a later saving; frequent reuse makes caching attractive.

Claude explicit checkpoint minimums are **Sonnet 5: 1,024; Opus 5: 512; Haiku 4.5: 4,096** prefix tokens. Opus 4.8 requires 1,024. The model tokenizer determines eligibility. Reuse the product manual below and inspect the returned write/read counters; a successful response can have zero cache usage.

This exercise uses explicit placement. Five-minute writes cost 1.25× input, one-hour writes 2×, and reads 0.1× for the selected models. [AWS caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).

**Explicit caching** uses a checkpoint you choose. **Implicit caching** lets the service reuse matching prefixes where supported; a hit still depends on eligibility and reuse before expiry. This exercise makes the boundary explicit.

For the selected Claude models, a five-minute write costs **1.25×**, a one-hour write **2×**, and a cache read **0.1×** the ordinary input price. With a prefix that ordinarily costs one unit, one five-minute write plus one read costs 1.35 units versus two uncached units; one one-hour write plus two reads costs 2.20 versus three. These illustrate prefix costs, excluding new input and output.

Price uncached input, cache writes, cache reads, and output once each. A high hit rate alone does not establish lower cost per solved task. Stable tools, policies, examples, and documents belong before the checkpoint; changing questions, timestamps, and retrieval results belong after it. With mixed Claude TTLs, one-hour checkpoints must precede five-minute checkpoints. A hit refreshes the reuse window.


### Choose a cache lifetime

The default five-minute lifetime suits frequent reuse; a one-hour lifetime can suit longer gaps between requests. Choose the lifetime before the static-prefix exercise. A one-hour write costs more, so compare total writes and reads.


In [ ]:
five_minutes = {"cachePoint": {"type": "default"}}  # Default TTL
one_hour = {"cachePoint": {"type": "default", "ttl": "1h"}}
CACHE_CHECKPOINT = five_minutes  # Choose one_hour to try the longer lifetime.


### Reuse a static document

Keep the product manual before the checkpoint and put each new question after it. Inspect the actual write/read counters; the first call may already find a warm cache.

`cachePoint` is part of the Converse request; there is no separate cache-creation call. Compare the raw `usage.cacheReadInputTokens`, `cacheWriteInputTokens`, and TTL breakdown in `cacheDetails` with the normalized meters. A missing raw field is shown as `None`; the pricing helper keeps uncached input, reads, writes, and output separate.


In [ ]:
PRODUCT_MANUAL = (REPO_ROOT / "02-optimization-playbook/data/product_manual.txt").read_text()
CACHE_MODEL = WORKHORSE
print({"model": CACHE_MODEL, "minimum_prefix_tokens": caps(CACHE_MODEL).cache_min_tokens,
       "manual_characters": len(PRODUCT_MANUAL)})
cache_questions = ["What is the return window for a defective item?", "Is the warranty transferable?"]
if supported(CACHE_MODEL, "cache_points"):
    prefix = [{"text": "Use this product manual as evidence:\n" + PRODUCT_MANUAL},
              CACHE_CHECKPOINT]
    for q in cache_questions:
        r = run_case(CACHE_MODEL, q, label="L04-static-prefix", system=prefix, max_tokens=1024)
        print({"Converse usage": {key: r["response"]["usage"].get(key) for key in
               ("inputTokens", "outputTokens", "cacheReadInputTokens", "cacheWriteInputTokens", "cacheDetails")}})
        show(r)


### Preserve an append-only conversation

This separate comparison uses a five-minute checkpoint at the latest user turn. `CACHE_CHECKPOINT` controls the static-document experiment above. Add each new user turn and retain the complete returned assistant content. Inspect the cache-read and cache-write counts to see whether the earlier history was reused. For longer conversations, retain an explicit checkpoint at a stable history boundary, within the four-checkpoint limit. Rewriting or summarizing earlier turns changes that history prefix.


In [ ]:
if supported(CACHE_MODEL, "cache_points"):
    # Append-only history with a single moving checkpoint; previous returned blocks stay intact.
    history = []
    for q in cache_questions:
        history.append({"role": "user", "content": [{"text": q}]})
        import copy
        sent = copy.deepcopy(history)
        sent[-1]["content"].append({"cachePoint": {"type": "default"}})
        r = run_case(CACHE_MODEL, label="L04-lookback", messages=sent,
                     system=[{"text": PRODUCT_MANUAL}], max_tokens=1024)
        print({"Converse usage": {key: r["response"]["usage"].get(key) for key in
               ("inputTokens", "outputTokens", "cacheReadInputTokens", "cacheWriteInputTokens", "cacheDetails")}})
        show(r)
        if r["stop_reason"] != "end_turn":
            print("Stop history replay: response did not complete normally.")
            break
        history.append(r["response"]["output"]["message"])


### Change the prefix — optional comparison

Set `RUN_PREFIX_CHANGE=True` after the static-prefix calls. Adding text before the manual changes the cached prefix. Compare write/read counters with the earlier requests; do not assume the entire request will miss or that an earlier entry was deleted.


In [ ]:
RUN_PREFIX_CHANGE = False
if RUN_PREFIX_CHANGE and supported(CACHE_MODEL, "cache_points"):
    changed_prefix = [
        {"text": "Reference edition 2. Use this product manual as evidence:\n" + PRODUCT_MANUAL},
        CACHE_CHECKPOINT,
    ]
    r = run_case(CACHE_MODEL, cache_questions[0], label="L04-changed-prefix",
                 system=changed_prefix, max_tokens=1024)
    print({"Converse usage": {key: r["response"]["usage"].get(key) for key in
           ("inputTokens", "outputTokens", "cacheReadInputTokens", "cacheWriteInputTokens", "cacheDetails")}})
    show(r)


### Optional GPT comparison: documented Responses caching

GPT-5.6 Luna supports implicit and explicit caching on **Responses**. The optional Converse calls below inspect returned usage without promising caching support or sending Claude checkpoints. For a supported cache-control experiment, use a developer-message breakpoint in the separate Responses exercise. Inline Bedrock Guardrails are unavailable on Responses.

Set `RUN_GPT_COMPARISON=1` for the provider comparison and `RUN_GPT_RESPONSES=1` for Responses. Keep API/profile and request context explicit. Credentials use the supplied AWS chain. [Luna model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-luna.html) · [Responses](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-responses-api.html).

The Responses exercise uses a **30-minute** explicit cache lifetime and a developer-message breakpoint. Luna requires at least **1,024 prefix tokens** and allows up to four cache writes per request. `store=False` controls response storage, separately from cache retention. Responses input usage includes cache categories: inspect `cached_tokens` and `cache_write_tokens` before estimating uncached input.


In [ ]:
if RUN_GPT:
    for q in cache_questions:
        show(run_case(GPT, q, label="L04-GPT-Converse-observation",
                      system=[{"text": PRODUCT_MANUAL}], max_tokens=1024, effort="low"))
else:
    print("Skipped optional GPT Converse usage observation.")

RUN_GPT_RESPONSES = os.environ.get("RUN_GPT_RESPONSES", "0") == "1"
if RUN_GPT_RESPONSES:
    if resolve_model(GPT).base_model_id not in {"openai.gpt-5.6-luna"}:
        raise ValueError("This dated Responses experiment is scoped to GPT-5.6 Luna.")
    from contextlib import nullcontext

    import httpx
    from aws_bedrock_token_generator import provide_token
    from openai import OpenAI

    from workshop_utils.bedrock import NormalizedUsage
    from workshop_utils.observability import langfuse_usage_details
    from workshop_utils.pacing import get_workshop_pacer
    responses = OpenAI(http_client=httpx.Client(event_hooks={"request": [get_workshop_pacer().httpx_request_hook]}), base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
                       api_key=provide_token(region=REGION), timeout=120, max_retries=2)
    for q in cache_questions:
        with (OBS.langfuse.start_as_current_observation(
            as_type="generation", name="Responses caching", model=GPT,
            input={"document": PRODUCT_MANUAL, "question": q},
        ) if OBS.langfuse else nullcontext()) as generation:
            r = responses.responses.create(
                model=GPT, store=False, max_output_tokens=1024, reasoning={"effort": "low"},
                input=[{"type": "message", "role": "developer", "content": [{
                    "type": "input_text", "text": PRODUCT_MANUAL,
                    "prompt_cache_breakpoint": {"mode": "explicit"}}]},
                    {"type": "message", "role": "user", "content": [{"type": "input_text", "text": q}]}],
                extra_body={"prompt_cache_options": {"mode": "explicit", "ttl": "30m"}},
            )
            if generation is not None:
                raw = r.usage.model_dump() if r.usage is not None else {}
                details = raw.get("input_tokens_details") or {}
                total, output = raw.get("input_tokens"), raw.get("output_tokens")
                read, write = details.get("cached_tokens"), details.get("cache_write_tokens")
                if all(type(value) is int and value >= 0 for value in (total, output, read, write)) and read + write <= total:
                    usage = NormalizedUsage(total - read - write, output, read, write, {"30m": write})
                    generation.update(output=r.output_text, usage_details=langfuse_usage_details(usage, model_id=GPT, api="responses"))
                else:
                    generation.update(output=r.output_text, level="WARNING", status_message="Complete usage was not returned; cost is unavailable.")
        print({"api": "Responses", "model": GPT, "usage": r.usage.model_dump() if r.usage is not None else None, "text": r.output_text})
    # Responses input includes cache categories; do not price it as Converse inputTokens.
else:
    print("Skipped optional Responses experiment.")


## 5. Reasoning Effort and Budgets — Lever 05

Higher effort is useful when evaluations show a quality improvement that justifies the additional cost and latency. Test that trade-off rather than choosing the highest available setting by default.

Claude Sonnet 5 and Opus 5 use `output_config.effort`; GPT-5.6 Luna uses `reasoning.effort`. Sonnet 5 and Opus 5 default to high, while Luna defaults to medium. Haiku 4.5 uses manual thinking rather than an effort parameter. Medium is a candidate to compare, not a universal best setting.

The next cell repeats four checkable support/math tasks at each selected effort level. `EFFORT_LEVELS` starts with low, medium, and high; `EFFORT_MODELS` starts with the workhorse and includes GPT when its comparison is enabled. Other values, including xhigh and max, depend on the selected model/API and need enough output room. The question, model, and output cap stay fixed within an effort comparison.

A response passes only when it matches the expected answer and finishes with `end_turn`. The expected answers are used for scoring, not sent as extra model instructions. These few examples teach the measurement workflow; add representative held-out cases before making a production choice.

**Thinking budgets and output limits differ.** Haiku's manual `budget_tokens` is a target inside the total output allowance. Sonnet 5 and Opus 5 use adaptive effort instead. GPT's output limit includes reasoning and visible output. Max effort never removes that limit.

Sources: [Claude effort](https://platform.claude.com/docs/en/build-with-claude/effort) · [Bedrock thinking budgets](https://docs.aws.amazon.com/bedrock/latest/userguide/claude-messages-extended-thinking.html) · [GPT reasoning](https://developers.openai.com/api/docs/guides/reasoning).

In Converse, these provider-specific controls live in `additionalModelRequestFields`: Claude uses `{"output_config": {"effort": "low"}}`, while GPT uses `{"reasoning": {"effort": "low"}}`. Edit the fields in the sweep below and inspect the actual request once per model/effort. Claude’s nested `output_config.effort` is distinct from the top-level Converse `outputConfig.textFormat` used for structured output. No manual `thinking.budget_tokens` request is made in this sweep.


In [ ]:
ARITHMETIC = "Calculate 17 * 23 * 19 - 1234. Return only the integer."
EXPECTED = str(17 * 23 * 19 - 1234)
EFFORT_TASKS = [
    {"id": "ticket-label", "question": "Classify as billing, technical, or returns. Return only the label. I was charged twice for the same order.",
     "expected": "billing"},
    {"id": "missing-evidence", "question": "Policy: returns require purchase within 30 days, a receipt, and original packaging. The customer bought the item 14 days ago and has the packaging; receipt status is unknown. Return only approve, deny, or ask_for_receipt.",
     "expected": "ask_for_receipt"},
    {"id": "invoice-total", "question": "An order costs $120 before a 10% discount. Shipping is free only if the discounted subtotal is at least $110; otherwise shipping is $5. Ignore tax. Return only the final total as an integer, without a currency symbol.",
     "expected": "113"},
    {"id": "arithmetic", "question": ARITHMETIC, "expected": EXPECTED},
]
EFFORT_LEVELS = ("low", "medium", "high")
EFFORT_MODELS = [WORKHORSE] + ([GPT] if RUN_GPT else [])
EFFORT_REPEATS = 2
EFFORT_OUTPUT_LIMIT = 2048
# Requirements for this small run; choose application-specific values before running.
MIN_PASS_RATE = 1.0
MIN_COMPLETION_RATE = 1.0
MAX_MEAN_LATENCY_MS = None  # Set a response-time budget if needed.

effort_rows = []
for model_id in EFFORT_MODELS:
    if not supported(model_id, "effort"):
        continue
    for effort in EFFORT_LEVELS:
        if effort not in caps(model_id).effort_values:
            print({"skipped_effort": effort, "model": model_id})
            continue
        # Native provider fields passed through Converse after capability validation.
        provider_fields = (
            {"output_config": {"effort": effort}}
            if resolve_model(model_id, region=REGION).provider == "anthropic"
            else {"reasoning": {"effort": effort}}
        )
        for task in EFFORT_TASKS:
            for repeat in range(EFFORT_REPEATS):
                r = run_case(model_id, task["question"],
                             label=f"L05-effort-{effort}-{task['id']}-{repeat}",
                             max_tokens=EFFORT_OUTPUT_LIMIT,
                             additional_model_request_fields=provider_fields)
                if task is EFFORT_TASKS[0] and repeat == 0:
                    print({"model": model_id, "additionalModelRequestFields":
                           r["request"]["additionalModelRequestFields"]})
                r.update({"effort": effort, "task_id": task["id"], "repeat": repeat,
                          "expected": task["expected"],
                          "correct": r["stop_reason"] == "end_turn"
                          and r["text"].strip() == task["expected"]})
                effort_rows.append(r)
                print({"model": model_id, "effort": effort, "task": task["id"],
                       "repeat": repeat, "answer": r["text"], "passed": r["correct"],
                       "cost_usd": r["cost_usd"], "latency_ms": r["latency_ms"]})


### Evaluate the quality gain before choosing more effort

The table counts every recorded attempt, including failed answers. **Cost per successful task** divides total model cost by successes; it remains unknown if pricing is missing or nothing succeeds. The plotted **cost per attempt** is a different measure and includes failed attempts in its average.

Set the quality and latency requirements before choosing a configuration. The result below applies only to this small run. Review the failed cases and price notes before drawing conclusions. Infrastructure and evaluator costs are separate from these model-call estimates.


In [ ]:
import statistics

import pandas as pd

effort_summary = []
for model_id in EFFORT_MODELS:
    for effort in EFFORT_LEVELS:
        rows = [r for r in effort_rows if r["model_id"] == model_id and r["effort"] == effort]
        if not rows:
            continue
        passed = sum(r["correct"] for r in rows)
        total_cost = sum(r["cost_usd"] for r in rows) if all(
            r["cost_usd"] is not None for r in rows) else None
        mean_latency = statistics.mean(r["latency_ms"] for r in rows)
        pass_rate = passed / len(rows)
        completion_rate = sum(r["stop_reason"] == "end_turn" for r in rows) / len(rows)
        effort_summary.append({
            "model_id": model_id, "effort": effort, "passed": passed, "attempts": len(rows),
            "pass_rate": pass_rate, "completion_rate": completion_rate, "total_cost_usd": total_cost,
            "mean_cost_per_attempt_usd": total_cost / len(rows) if total_cost is not None else None,
            "cost_per_success_usd": total_cost / passed if total_cost is not None and passed else None,
            "mean_latency_ms": mean_latency,
            "median_latency_ms": statistics.median(r["latency_ms"] for r in rows),
            "mean_output_tokens": statistics.mean(r["output_tokens"] for r in rows),
            "meets_requirements": pass_rate >= MIN_PASS_RATE and completion_rate >= MIN_COMPLETION_RATE
            and (MAX_MEAN_LATENCY_MS is None or mean_latency <= MAX_MEAN_LATENCY_MS),
        })

display(pd.DataFrame(effort_summary))
priced_candidates = [r for r in effort_summary
                     if r["meets_requirements"] and r["cost_per_success_usd"] is not None]
if priced_candidates:
    choice = min(priced_candidates, key=lambda r: (r["cost_per_success_usd"], r["mean_latency_ms"]))
    print({"lowest_cost_passing_configuration_in_this_run": choice})
else:
    print("No fully priced configuration meets the selected requirements in this run.")


### Read the trade-off

Both plots use linear axes and only measurements from your run. Follow each model across effort levels. A flat quality curve with rising cost or latency provides no reason to prefer the higher setting on these tasks. If quality improves, judge whether the additional cost and waiting time fit the application. Missing prices are omitted from the cost plot, not treated as zero.


In [ ]:
import matplotlib.pyplot as plt

if effort_summary:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for model_id in EFFORT_MODELS:
        rows = [r for r in effort_summary if r["model_id"] == model_id]
        priced = [r for r in rows if r["mean_cost_per_attempt_usd"] is not None]
        for axis, points, metric in (
            (axes[0], priced, "mean_cost_per_attempt_usd"),
            (axes[1], rows, "mean_latency_ms"),
        ):
            if not points:
                continue
            axis.plot([r[metric] for r in points], [100 * r["pass_rate"] for r in points],
                      marker="o", label=model_id)
            for r in points:
                axis.annotate(r["effort"], (r[metric], 100 * r["pass_rate"]),
                              xytext=(4, 5), textcoords="offset points", fontsize=8)
    axes[0].set_xlabel("Mean model cost per task attempt (USD)")
    axes[1].set_xlabel("Mean task latency (ms)")
    axes[0].set_ylabel("Pass rate (%)")
    for axis in axes:
        axis.set_ylim(-5, 105)
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend(fontsize=7, loc="best")
    fig.suptitle("Does higher effort earn its cost and latency?")
    plt.tight_layout()
    plt.show()
else:
    print("Run the effort comparison before plotting results.")


### Retry with more effort when needed

If the arithmetic check fails, allow one retry at higher effort. Include both attempts when comparing cost.

This cell reuses the sweep’s arithmetic question and expected answer. Acceptance requires the exact answer and `end_turn`; otherwise the second attempt uses high effort. Each call allows up to 2,048 output tokens.


In [ ]:
# A bounded cascade: only an external failed check triggers another attempt.
if supported(WORKHORSE, "effort"):
    attempts = []
    for effort in ("low", "high"):
        r = run_case(WORKHORSE, ARITHMETIC, label="L05-cascade-" + effort,
                     max_tokens=2048, effort=effort)
        attempts.append(r)
        if r["stop_reason"] == "end_turn" and r["text"].strip() == EXPECTED:
            break
    print({"attempts": len(attempts), "solved": attempts[-1]["stop_reason"] == "end_turn" and attempts[-1]["text"].strip() == EXPECTED,
           "cost": sum(r["cost_usd"] for r in attempts) if all(r["cost_usd"] is not None for r in attempts) else None})

### Optional draft → critique → revise

The **Self-Refine pattern** separates three jobs: generate a draft, name its flaws, and revise it. This exercise uses the original October double-charge example. Read the draft and revision against the supplied policy, checking **empathy, a concrete next step, accuracy, and tone**. The revision must contain only the customer-facing reply; critique and change notes belong outside that output. Inspect the raw revision for both policy compliance and this output boundary.

Set `RUN_SELF_REFINE=1` to run the cell. The notebook uses a small model for the draft and revision and a workhorse for critique—a variation on same-model Self-Refine. Output caps remain 512, 1,024, and 512 tokens respectively. The final summary compares the first draft's model cost/latency with the full three-call workflow; judge whether the revision's quality improvement justifies the extra work.


In [ ]:
RUN_REFINE = os.environ.get("RUN_SELF_REFINE", "0") == "1"
if RUN_REFINE:
    policy = "Do not claim a refund has been issued. Request the order reference and explain that billing must verify the duplicate charge."
    refine_started = time.perf_counter()
    draft = run_case(SMALL, 'Write a support reply to: "I was charged twice for October. Fix it."',
                     label="L05-draft", max_tokens=512)
    critique_prompt = (
        f"Policy: {policy}\nHere is a draft reply: {draft['text']}\n"
        "Critique it, one bullet each: empathy · concrete next step · accuracy\n"
        "(promises nothing we can't verify) · tone. Say what's wrong; do NOT rewrite."
    )
    critique = run_case(WORKHORSE, critique_prompt, label="L05-critique", max_tokens=1024)
    revision_prompt = (
        f"Policy: {policy}\nDraft: {draft['text']}\nCritique: {critique['text']}\n"
        "Rewrite to address every point; keep what the critique said was already good. "
        "Return only the customer-facing reply. Do not include critique, change notes, "
        "a Changes made section, or any other explanation outside the reply."
    )
    revised = run_case(SMALL, revision_prompt, label="L05-revise", max_tokens=512)
    workflow_latency_ms = (time.perf_counter() - refine_started) * 1000
    refinement_rows = [draft, critique, revised]
    for r in refinement_rows:
        show(r)
    workflow_cost = sum(r["cost_usd"] for r in refinement_rows) if all(
        r["cost_usd"] is not None for r in refinement_rows) else None
    print({"draft_cost_usd": draft["cost_usd"], "draft_latency_ms": draft["latency_ms"],
           "workflow_cost_usd": workflow_cost, "workflow_latency_ms": workflow_latency_ms,
           "calls": len(refinement_rows)})
else:
    print("Skipped optional self-refinement experiment.")


### Additional prompt patterns

The following are prompt templates from the walkthrough. Fill in the task and evidence before trying them; these Markdown cells do not make model calls. The preceding self-refinement cell is the executable example.


### Prompt template: Chain-of-Draft — keep intermediate notes concise

[Chain-of-Draft](https://arxiv.org/abs/2502.18600) asks for brief intermediate notes before the final answer. Append the following suffix to a task prompt when trying this pattern with thinking off:

```text
Think step by step, but keep each thinking step to a minimum draft of
5 words at most. Return the final answer after a separator: ####.
```

Compare correctness, total token usage, and latency against the same task without the suffix. The instruction shapes visible output; it does not set the model's internal thinking budget. Shorter notes can lose useful detail on harder problems, so check quality before adopting the pattern.


### Prompt template: Verbalized Sampling — explore more diverse responses

[Verbalized Sampling](https://arxiv.org/abs/2510.01171) asks for several responses with probability estimates. Try it when brainstorming, creative variants, or synthetic examples keep converging on the same answer:

```text
Generate 5 distinct responses, each with a probability estimate of how
likely you'd produce it. Then return one sampled according to those probabilities.
```

Append this to the actual task, such as generating coffee-shop taglines. Inspect the candidates for both diversity and relevance. The probabilities are model-generated estimates, not calibrated confidence or proof of correctness; use application-side sampling when you need reproducible selection. Generating several candidates also costs more tokens than returning one.


### Prompt template: Chain-of-Verification — check facts in a fresh context

[Chain-of-Verification](https://arxiv.org/abs/2309.11495) follows **draft → verification questions → independent answers → revision**. The Enterprise SLA/warranty example makes the evidence requirement explicit:

```text
# Stage 1 — Baseline draft
What's our SLA and warranty policy for the Enterprise tier?

# Stage 2 — Plan verification questions
Draft: {BASELINE_DRAFT}
List each factual claim; write ONE verification question per claim.

# Stage 3 — Answer each question in a CLEAN context, once per question
{RETRIEVED_DOCS}
What uptime SLA does the Enterprise tier provide, and what conditions apply?
Answer using only these documents. If a fact is missing, say it is not established.

# Stage 4 — Revise
Draft: {BASELINE_DRAFT}
Verified facts: {Q1→A1, Q2→A2, …}
Rewrite to match the verified facts. Correct contradictions; add no new claims.
```

Replace `{RETRIEVED_DOCS}` with actual SLA and warranty excerpts. Each Stage 3 call receives only the evidence and its verification question, without the draft or other verification answers. This helps the verifier assess the source rather than repeat the draft's claims.

Count the draft, planning, verification, retrieval, and revision work when comparing cost and elapsed time. A shared policy prefix may be reusable through [prompt caching](https://catalog.us-east-1.prod.workshops.aws/workshops/60d21a0a-c56f-47aa-9e5d-45181cd42507/en-US/3-optimization-playbook/31-low-effort/314-prompt-caching).


### Bound the application task — optional live

Use the selected Opus 5 model with at most two generation attempts and a shared 2,048-output-token allowance. The next request is limited to the remaining output allowance, and execution stops as soon as the exact arithmetic check passes and the response finishes with `end_turn`. Track input tokens, tool calls and evaluator charges separately: this is an application output/attempt limit, not a whole-account spending cap. Set `RUN_BOUNDED_TASK=1` to execute.

Each call is capped at the smaller of 1,024 tokens and the remaining shared allowance. Run the effort sweep first to define the arithmetic question and expected answer. These limits apply to this optional task; the sweep, retry cascade, and self-refinement are separate experiments. Extensions with tools or model-based checkers need their own limits and input/cost accounting.

**These application counters are not Claude's native Task Budgets feature.** Native `output_config.task_budget` is an advisory budget across an agentic loop on supported models/APIs. The cell below enforces its own generation-attempt and reported-output limits; it does not enable that native beta. See [Task Budgets](https://platform.claude.com/docs/en/build-with-claude/task-budgets) for the separate concept.


In [ ]:
RUN_BOUNDED_TASK = os.environ.get("RUN_BOUNDED_TASK", "0") == "1"
if RUN_BOUNDED_TASK and supported(DEEP, "effort"):
    MAX_TASK_ATTEMPTS, TASK_OUTPUT_ALLOWANCE = 2, 2048
    task_attempts = []
    for effort in ("low", "high"):
        remaining = TASK_OUTPUT_ALLOWANCE - sum(row["output_tokens"] for row in task_attempts)
        if len(task_attempts) >= MAX_TASK_ATTEMPTS or remaining <= 0:
            break
        row = run_case(DEEP, ARITHMETIC, label="L05-bounded-"+effort,
                       max_tokens=min(1024, remaining), effort=effort)
        task_attempts.append(row)
        if row["text"].strip() == EXPECTED and row["stop_reason"] == "end_turn":
            break
    print({"model": DEEP, "attempts": len(task_attempts),
           "output_tokens": sum(row["output_tokens"] for row in task_attempts),
           "solved": bool(task_attempts and task_attempts[-1]["text"].strip() == EXPECTED
                          and task_attempts[-1]["stop_reason"] == "end_turn"),
           "attempt_limit": MAX_TASK_ATTEMPTS, "output_allowance": TASK_OUTPUT_ALLOWANCE})
else:
    print("Skipped optional bounded application task.")


### Compare cost, latency, and quality

Use a clear prompt at an appropriate effort level as the baseline. Add a technique to solve a measured problem, then compare the **whole workflow**:

| Change | Cost and latency trade-off | Quality check |
|---|---|---|
| Higher effort | May use more reasoning tokens and time | Does it solve cases the lower setting misses? |
| Chain-of-Draft | Can shorten visible output; measure total usage | Does the concise answer remain correct and complete? |
| Verbalized Sampling | More candidates usually mean more output tokens | Are the responses more varied while remaining relevant? |
| Self-Refine | Adds critique and revision calls | Does the revision improve policy compliance, tone, and usefulness? |
| Chain-of-Verification | Adds verification calls and possibly retrieval | Are the final factual claims supported by the evidence? |

Track success rate, **cost per solved task**, and end-to-end latency. Include every attempt and check; a cheaper final call alone does not establish a cheaper workflow. Keep a change when its quality benefit meets your cost and response-time requirements.

---

For automated prompt optimization, use the [HIGH notebook](03-high-effort.ipynb), **Lever 14**.


## Quality check and measurement

Use the returned usage for each executed call. Prices are dated estimates; reconcile SDK retries and service telemetry before a billing claim. No TTFT is reported for non-streaming calls. Optimizer/evaluator and infrastructure costs are separate from serving cost. A few passing examples are a smoke check, not general quality preservation.

For each lever, write one sentence explaining the mechanism you observed: fewer output tokens, lower token rates, or more cached input. Then identify a counterexample that could erase that saving, such as truncation, a cache miss, or a failed small-model answer. This turns a table of numbers into a reusable decision.

The optional cell below sends one existing response span to `bedrock-agentcore.evaluate`, with its expected answer attached to the exact session and trace. This is a response correctness check, not an agent invocation or a full tool-trajectory evaluation. Inspect `evaluationResults` and keep evaluator token usage separate from the model-call totals.


In [ ]:
if RUN_EVAL and selection_rows:
    row = selection_rows[0]
    if row["span"] is not None:
        span = row["span"]
        evaluator_id = "Builtin.Correctness"
        session_id = span["attributes"]["session.id"]
        trace_id = span["traceId"]
        expected = LABELLED_TICKETS[0][1]
        # Check the single locally generated response observation before sending it.
        if not isinstance(session_id, str) or not session_id:
            raise ValueError("The response span needs an explicit session ID.")
        for value, size in ((trace_id, 32), (span["spanId"], 16)):
            if (not isinstance(value, str) or len(value) != size
                    or any(c not in "0123456789abcdef" for c in value) or int(value, 16) == 0):
                raise ValueError("The response span needs valid nonzero trace/span IDs.")
        if not span["attributes"].get("workshop.response_only") or not span["attributes"].get("gen_ai.task.output"):
            raise ValueError("Supply the recorded response observation, including its answer.")
        if not isinstance(expected, str) or not 1 <= len(expected) <= 100000:
            raise ValueError("Expected response text must contain 1..100000 characters.")
        evaluate_shape = AGENTCORE.meta.service_model.operation_model("Evaluate").input_shape
        if "evaluationReferenceInputs" not in evaluate_shape.members:
            raise RuntimeError("Install the workshop SDK pins to send evaluation ground truth.")

        # One native AgentCore Evaluate call; it evaluates the existing answer.
        evaluation = AGENTCORE.evaluate(
            evaluatorId=evaluator_id,
            evaluationInput={"sessionSpans": [span]},
            evaluationTarget={"traceIds": [trace_id]},
            evaluationReferenceInputs=[{
                "context": {"spanContext": {"sessionId": session_id, "traceId": trace_id}},
                "expectedResponse": {"text": expected},
            }],
        )
        results = evaluation.get("evaluationResults")
        if not isinstance(results, list) or not results:
            raise RuntimeError("Evaluate returned no results.")
        for result in results:
            if result.get("errorCode") or result.get("errorMessage"):
                raise RuntimeError(f"Evaluator error: {result}")
            if result.get("ignoredReferenceInputFields"):
                raise RuntimeError(f"Evaluator ignored supplied ground truth: {result}")
            context = result.get("context", {}).get("spanContext", {})
            if (result.get("evaluatorId") != evaluator_id
                    or context.get("sessionId") != session_id or context.get("traceId") != trace_id):
                raise RuntimeError(f"Unexpected evaluator/session/trace: {result}")
            if result.get("value") is None and result.get("label") is None:
                raise RuntimeError(f"Evaluate returned no score or label: {result}")
        print(results)  # Evaluator tokenUsage is separate from model-serving usage.
    else:
        print("Optional AgentCore response check needs a recorded text response.")
else:
    print("Optional AgentCore response check not run.")
print({"recorded_model_calls": len(RUN_ROWS), "flush_completed": OBS.flush()})


## What you measured

01 model selection · 02 prompt design, provider by provider · 03 output and sampling controls · 04 prompt caching, provider by provider · 05 reasoning effort and budgets. Preserve experiment inputs and actual results before changing configuration. Continue with [MEDIUM](02-medium-effort.ipynb), or [HIGH](03-high-effort.ipynb) for **Lever 14 automated prompt optimization**.

## References

- [Converse API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
- [Claude prompting](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices)
- [GPT prompting](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6)
- [Structured output](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html)
- [Prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html)
- [Claude effort](https://platform.claude.com/docs/en/build-with-claude/effort)
- [Token burndown](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas-token-burndown.html)
- [AgentCore Evaluate API](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/evaluate.html)
